# ONNX-Tool Fusion Validation Template (YOLOv5 Conv+Silu)

## Overview
This notebook provides a reusable local verification workflow to:
1. Define YOLOv5-style `Conv + Sigmoid + Mul -> Conv_Silu` fusion in ONNX-Tool.
2. Validate transformation correctness with fast local tensor checks.
3. Optionally run end-to-end output checks.
4. Export fused models for analysis/deployment.

## Demo Model
- Preferred PT path: `data/yolov5/yolov5s.pt`
- Fallback PT path: `data/public/yolov5/yolov5s.pt`
- ONNX output: `data/yolov5/yolov5s.onnx` (fallback: `data/public/yolov5/yolov5s.onnx`)


## Setup


In [7]:
import copy
from pathlib import Path

import numpy as np
import onnx

from onnx_tool import loadmodel
from onnx_tool.fusion import FusionPattern


## Paths and Optional PT->ONNX Export

The notebook first tries your requested local path (`data/yolov5/...`). If not found, it falls back to `data/public/yolov5/...`.

If `yolov5s.onnx` does not exist, the cell attempts export with `ultralytics`. If export fails, export ONNX manually and rerun.


In [8]:
def pick_existing_path(candidates):
    for c in candidates:
        if c.exists():
            return c
    return candidates[0]

pt_candidates = [
    Path('data/public/yolov5/yolov5s.pt'),
]
onnx_candidates = [
    Path('data/public/yolov5/yolov5s.onnx'),
]

pt_path = pick_existing_path(pt_candidates)
onnx_path = pick_existing_path(onnx_candidates)
onnx_path.parent.mkdir(parents=True, exist_ok=True)

print('Selected PT path:', pt_path)
print('Selected ONNX path:', onnx_path)
print('PT exists:', pt_path.exists())
print('ONNX exists:', onnx_path.exists())

if not onnx_path.exists():
    try:
        from ultralytics import YOLO
        model_ultra = YOLO(str(pt_path))
        export_result = model_ultra.export(format='onnx', opset=12, imgsz=640)
        print('Export result:', export_result)
    except Exception as e:
        raise RuntimeError(
            'ONNX file is missing and automatic export failed. '
            'Please export yolov5s.onnx manually, then rerun. '
            f'Original error: {e}'
        )


Selected PT path: data\public\yolov5\yolov5s.pt
Selected ONNX path: data\public\yolov5\yolov5s.onnx
PT exists: False
ONNX exists: True


## Load Model and Build Compute Graph


In [9]:
model_path = str(onnx_path)
model = loadmodel(model_path)
full_graph = model.graph
cg = full_graph.get_compute_graph()

print('Full graph nodes:', len(full_graph.nodemap))
print('Compute graph nodes:', len(cg.nodemap))


Full graph nodes: 292
Compute graph nodes: 258


## Define and Search YOLOv5 Conv+Sigmoid+Mul Pattern


In [10]:
ConvSilu_pattern = [
    {
        'name': 'conv_0',
        'op': 'Conv',
        'attrs': [],
        'inport': [],
        # Conv output goes to Sigmoid and also directly to Mul (skip branch).
        'outport': [[0, 'sigmoid_1', 0], [0, 'mul_2', -1]],
    },
    {
        'name': 'sigmoid_1',
        'op': 'Sigmoid',
        'attrs': [],
        'inport': [[0, 'conv_0', 0]],
        'outport': [[0, 'mul_2', -1]],
    },
    {
        'name': 'mul_2',
        'op': 'Mul',
        'attrs': [],
        # Use wildcard input ports because Mul input order may vary across exports.
        'inport': [[-1, 'conv_0', 0], [-1, 'sigmoid_1', 0]],
        'outport': [],
    },
]

pattern = FusionPattern(ConvSilu_pattern)
found_nodes = pattern.search_pattern(cg)
print('Found Conv+Sigmoid+Mul modules:', len(found_nodes))
for i, nodes in enumerate(found_nodes[:10]):
    print(f'  {i+1}: {nodes}')


Found Conv+Sigmoid+Mul modules: 57
  1: ['/model.0/conv/Conv', '/model.0/act/Sigmoid', '/model.0/act/Mul']
  2: ['/model.1/conv/Conv', '/model.1/act/Sigmoid', '/model.1/act/Mul']
  3: ['/model.2/cv1/conv/Conv', '/model.2/cv1/act/Sigmoid', '/model.2/cv1/act/Mul']
  4: ['/model.2/cv2/conv/Conv', '/model.2/cv2/act/Sigmoid', '/model.2/cv2/act/Mul']
  5: ['/model.2/m/m.0/cv1/conv/Conv', '/model.2/m/m.0/cv1/act/Sigmoid', '/model.2/m/m.0/cv1/act/Mul']
  6: ['/model.2/m/m.0/cv2/conv/Conv', '/model.2/m/m.0/cv2/act/Sigmoid', '/model.2/m/m.0/cv2/act/Mul']
  7: ['/model.2/cv3/conv/Conv', '/model.2/cv3/act/Sigmoid', '/model.2/cv3/act/Mul']
  8: ['/model.3/conv/Conv', '/model.3/act/Sigmoid', '/model.3/act/Mul']
  9: ['/model.4/cv1/conv/Conv', '/model.4/cv1/act/Sigmoid', '/model.4/cv1/act/Mul']
  10: ['/model.4/cv2/conv/Conv', '/model.4/cv2/act/Sigmoid', '/model.4/cv2/act/Mul']


## Export and Verify One Local Conv+Sigmoid+Mul Subgraph

This cell exports only the first matched module as a standalone ONNX model for quick local validation.


In [ ]:
if not found_nodes:
    raise RuntimeError('No Conv+Sigmoid+Mul module found. Run pattern search cell first.')

first_nodes = found_nodes[0]
print('Export first module nodes:', first_nodes)

# Use a fresh full graph for stable export.
sub_model_src = loadmodel(str(onnx_path))
sub_graph_src = sub_model_src.graph
sub_graph_proto = sub_graph_src.get_onnxgraph_by_nodenames(first_nodes)
if sub_graph_proto is None:
    raise RuntimeError('Failed to build subgraph proto.')

sub_model_proto = onnx.helper.make_model(sub_graph_proto, producer_name='onnx_tool_subgraph_demo')
sub_model_proto.ir_version = sub_model_src.mproto.ir_version
del sub_model_proto.opset_import[:]
for opset in sub_model_src.mproto.opset_import:
    sub_model_proto.opset_import.append(opset)

sub_path = onnx_path.with_name('yolov5s_first_convsilu_subgraph.onnx')
onnx.save(sub_model_proto, str(sub_path))
print('Saved subgraph model:', sub_path)

# Quick local verification on the exported subgraph itself.
sub_loaded = loadmodel(str(sub_path))
sub_g = sub_loaded.graph
sub_input = sub_g.input[0]
sub_shape_spec = sub_g.tensormap[sub_input].shape
sub_test_shape = [d if isinstance(d, int) and d > 0 else 1 for d in sub_shape_spec]
if len(sub_test_shape) == 3:
    sub_test_shape.insert(1, 3)

np.random.seed(7)
sub_probe = np.zeros(sub_test_shape, dtype=np.float32)
sub_x = np.random.randn(*sub_test_shape).astype(np.float32)

sub_g.shape_infer({sub_input: sub_probe})
_ = sub_g.value_infer({sub_input: sub_x})

conv_out_name = sub_g.nodemap[first_nodes[0]].output[0]
sig_out_name = sub_g.nodemap[first_nodes[1]].output[0]
mul_out_name = sub_g.nodemap[first_nodes[2]].output[0]

conv_out = sub_g.tensormap[conv_out_name].numpy
sig_out = sub_g.tensormap[sig_out_name].numpy
mul_out = sub_g.tensormap[mul_out_name].numpy

sig_ref = 1.0 / (1.0 + np.exp(-np.clip(conv_out, -60.0, 60.0)))
mul_ref = conv_out * sig_ref

sig_max_diff = float(np.max(np.abs(sig_out - sig_ref)))
mul_max_diff = float(np.max(np.abs(mul_out - mul_ref)))
print('Subgraph input:', sub_input, 'shape:', sub_test_shape)
print('Sigmoid check max abs diff:', sig_max_diff)
print('Mul check max abs diff:', mul_max_diff)
print('Sigmoid allclose:', np.allclose(sig_out, sig_ref, rtol=1e-5, atol=1e-5))
print('Mul allclose:', np.allclose(mul_out, mul_ref, rtol=1e-5, atol=1e-5))


Export first module nodes: ['/model.0/conv/Conv', '/model.0/act/Sigmoid', '/model.0/act/Mul']
Saved subgraph model: data\public\yolov5\yolov5s_first_convsilu_subgraph.onnx


## Fuse The Exported Subgraph and Compare Outputs

Run Conv+Sigmoid+Mul fusion on the exported subgraph, then compare final output with the original subgraph.


In [ ]:
# 1) Load original subgraph and build a fused copy
sub_orig_model = loadmodel(str(sub_path))
sub_orig_graph = sub_orig_model.graph

sub_fused_model = loadmodel(str(sub_path))
sub_fused_graph = sub_fused_model.graph
sub_sets = pattern.search_pattern(sub_fused_graph)
print('Matched modules in subgraph:', len(sub_sets))

if not sub_sets:
    raise RuntimeError('No Conv+Sigmoid+Mul pattern found in exported subgraph.')

fuse_conv_silu(sub_fused_graph, sub_sets, fused_op='Conv_Silu', domain='ai.custom')
sub_fused_path = sub_path.with_name(sub_path.stem + '_fused.onnx')
sub_fused_graph.save_model(str(sub_fused_path), rawmodel=sub_fused_model.mproto, no_shape=True)
print('Saved fused subgraph:', sub_fused_path)

# 2) Compare final outputs on the same input
sub_input_name = sub_orig_graph.input[0]
sub_shape_spec = sub_orig_graph.tensormap[sub_input_name].shape
sub_test_shape = [d if isinstance(d, int) and d > 0 else 1 for d in sub_shape_spec]
if len(sub_test_shape) == 3:
    sub_test_shape.insert(1, 3)

np.random.seed(11)
sub_probe = np.zeros(sub_test_shape, dtype=np.float32)
sub_x = np.random.randn(*sub_test_shape).astype(np.float32)

sub_orig_graph.shape_infer({sub_input_name: sub_probe})
sub_fused_graph.shape_infer({sub_input_name: sub_probe})
y_orig = sub_orig_graph.value_infer({sub_input_name: sub_x})
y_fused = sub_fused_graph.value_infer({sub_input_name: sub_x})

print('Output count:', len(y_orig), len(y_fused))
if y_orig and y_fused:
    diff = np.abs(y_orig[0] - y_fused[0])
    print('Subgraph output max abs diff:', float(np.max(diff)))
    print('Subgraph output mean abs diff:', float(np.mean(diff)))
    print('Subgraph output allclose:', np.allclose(y_orig[0], y_fused[0], rtol=1e-5, atol=1e-5))


## Reusable Local Verification Template

Template strategy:
1. Build fresh original/fused compute graphs from the same model.
2. Fuse YOLOv5-style `Conv+Sigmoid+Mul` only on the fused branch.
3. Compare local tensors around transformed nodes first (fast).
4. Use end-to-end output checks as a final gate.


In [5]:
def fuse_conv_silu(graph_obj, node_sets, fused_op='Conv_Silu', domain='ai.custom'):
    for nodes in node_sets:
        base = nodes[0]
        fused_name = f'{base}_fused'
        graph_obj.fuse_subgraph_node_names(
            nodes,
            fused_op,
            fused_name,
            keep_attr=True,
            nodedomain=domain,
        )
    graph_obj.graph_reorder_nodes()
    return graph_obj


def build_local_tensor_list(graph_obj, node_sets, max_items=64):
    names = []
    for nodes in node_sets:
        if not nodes:
            continue
        boundary = nodes[-1]
        fallback = nodes[0]
        if boundary in graph_obj.nodemap:
            names.extend(graph_obj.nodemap[boundary].output)
        elif fallback in graph_obj.nodemap:
            names.extend(graph_obj.nodemap[fallback].output)

    uniq = []
    seen = set()
    for n in names:
        if n not in seen:
            uniq.append(n)
            seen.add(n)
    return uniq[:max_items]


def local_verify_template(orig_graph, fused_graph, input_name, test_input, probe_input, tensor_names, rtol=1e-5, atol=1e-5):
    orig_graph.shape_infer({input_name: probe_input})
    fused_graph.shape_infer({input_name: probe_input})

    _ = orig_graph.value_infer({input_name: test_input})
    _ = fused_graph.value_infer({input_name: test_input})

    report = []
    for t in tensor_names:
        if t not in orig_graph.tensormap or t not in fused_graph.tensormap:
            continue
        a = orig_graph.tensormap[t].numpy
        b = fused_graph.tensormap[t].numpy
        if a is None or b is None:
            continue
        if a.shape != b.shape:
            report.append((t, False, float('inf'), f'shape mismatch {a.shape} vs {b.shape}'))
            continue
        ok = np.allclose(a, b, rtol=rtol, atol=atol)
        md = float(np.max(np.abs(a - b))) if a.size else 0.0
        report.append((t, bool(ok), md, ''))

    all_ok = all(x[1] for x in report) if report else False
    return all_ok, report


def run_conv_silu_local_verification(model_path, pattern, fused_op='Conv_Silu', domain='ai.custom', rtol=1e-5, atol=1e-5, max_tensors=64, seed=42):
    model_orig = loadmodel(str(model_path))
    orig_cg = model_orig.graph.get_compute_graph()
    orig_cg.graph_reorder_nodes()

    model_fused = loadmodel(str(model_path))
    fused_cg = model_fused.graph.get_compute_graph()
    node_sets = pattern.search_pattern(fused_cg)
    fuse_conv_silu(fused_cg, node_sets, fused_op=fused_op, domain=domain)

    input_name = orig_cg.input[0]
    shape_spec = orig_cg.tensormap[input_name].shape
    test_shape = [d if isinstance(d, int) and d > 0 else 1 for d in shape_spec]
    if len(test_shape) == 2:
        test_shape.extend([640, 640])
    elif len(test_shape) == 3:
        test_shape.insert(1, 3)

    np.random.seed(seed)
    probe = np.zeros(test_shape, dtype=np.float32)
    x = np.random.randn(*test_shape).astype(np.float32)

    local_tensors = build_local_tensor_list(orig_cg, node_sets, max_items=max_tensors)
    ok, report = local_verify_template(orig_cg, fused_cg, input_name, x, probe, local_tensors, rtol=rtol, atol=atol)

    return {
        'ok': ok,
        'report': report,
        'local_tensors': local_tensors,
        'input_name': input_name,
        'test_shape': test_shape,
        'probe': probe,
        'x': x,
        'orig_cg': orig_cg,
        'fused_cg': fused_cg,
        'node_sets': node_sets,
        'model_path': str(model_path),
    }


## Run Fusion on Compute Graph and Validate Locally


In [ ]:
result = run_conv_silu_local_verification(onnx_path, pattern)

orig_cg = result['orig_cg']
fused_cg = result['fused_cg']
node_sets = result['node_sets']
input_name = result['input_name']
probe = result['probe']
x = result['x']

print('Model path:', result['model_path'])
print('Found Conv+Sigmoid+Mul modules:', len(node_sets))
print('Input tensor name:', input_name)
print('Test input shape:', result['test_shape'])
print('Local tensor verification tensors:', len(result['local_tensors']))
print('Local verification pass:', result['ok'])

for t, pass_flag, md, note in result['report'][:20]:
    print(f'  {t}: {"PASS" if pass_flag else "FAIL"}, max_diff={md:.10f} {note}')


## Minimal Copy-Paste Template (Other Models)

Change only `template_model_path` and `template_pattern_def`.


In [ ]:
# Minimal local verification template for any fusion pattern.
# Required: helper functions already defined above in this notebook.

template_model_path = onnx_path  # e.g. Path('path/to/your_model.onnx')
template_pattern_def = [
    {
        'name': 'conv_0',
        'op': 'Conv',
        'attrs': [],
        'inport': [],
        'outport': [[0, 'sigmoid_1', 0], [0, 'mul_2', -1]],
    },
    {
        'name': 'sigmoid_1',
        'op': 'Sigmoid',
        'attrs': [],
        'inport': [[0, 'conv_0', 0]],
        'outport': [[0, 'mul_2', -1]],
    },
    {
        'name': 'mul_2',
        'op': 'Mul',
        'attrs': [],
        'inport': [[-1, 'conv_0', 0], [-1, 'sigmoid_1', 0]],
        'outport': [],
    },
]
template_fused_op = 'Conv_Silu'
template_domain = 'ai.custom'

template_pattern = FusionPattern(template_pattern_def)
template_result = run_conv_silu_local_verification(
    model_path=template_model_path,
    pattern=template_pattern,
    fused_op=template_fused_op,
    domain=template_domain,
    rtol=1e-5,
    atol=1e-5,
    max_tensors=64,
    seed=42,
)

print('Template model:', template_result['model_path'])
print('Matched modules:', len(template_result['node_sets']))
print('Local tensors checked:', len(template_result['local_tensors']))
print('Verification pass:', template_result['ok'])
for t, pass_flag, md, note in template_result['report'][:10]:
    print(f"  {t}: {'PASS' if pass_flag else 'FAIL'}, max_diff={md:.10f} {note}")


## Optional End-to-End Check on Final Outputs


In [ ]:
orig_cg.shape_infer({input_name: probe})
fused_cg.shape_infer({input_name: probe})

y0 = orig_cg.value_infer({input_name: x})
y1 = fused_cg.value_infer({input_name: x})

print('Output count:', len(y0), len(y1))
if y0 and y1:
    same = np.allclose(y0[0], y1[0], rtol=1e-5, atol=1e-5)
    md = float(np.max(np.abs(y0[0] - y1[0])))
    print('Output[0] allclose:', same)
    print('Output[0] max abs diff:', md)


## Export Fused Models

- `yolov5s_fused.onnx`: compute-graph export (analysis-oriented)
- `yolov5s_fused_full.onnx`: full-graph export (deployment-oriented)


In [ ]:
# Export 1: compute-graph fused model
output_cg = onnx_path.with_name('yolov5s_fused.onnx')
fused_cg.save_model(str(output_cg), rawmodel=loadmodel(str(onnx_path)).mproto)
print('Saved compute-graph fused model:', output_cg)


In [ ]:
# Export 2: full-graph fused model (recommended for runtime deployment)
model_export = loadmodel(str(onnx_path))
full_export_graph = model_export.graph
full_sets = pattern.search_pattern(full_export_graph)
fuse_conv_silu(full_export_graph, full_sets, fused_op='Conv_Silu', domain='ai.custom')

output_full = onnx_path.with_name('yolov5s_fused_full.onnx')
full_export_graph.save_model(str(output_full), rawmodel=model_export.mproto, no_shape=True)
print('Saved full-graph fused model:', output_full)


## Common Issues and Fixes

1. Incomplete model after `get_compute_graph()` export
- Symptom: missing producer for shape tensors in runtime.
- Cause: compute graph intentionally removes shape-only branches.
- Fix: export deployment model from a fresh full graph (`model.graph`), not from `cg`.

2. Slow full-model `value_infer`
- Use local verification first: compare only tensors around transformed subgraphs.
- Then run a small number of end-to-end checks as final guard.


## QA

### Q1: Should I validate the whole model every time?
No. For expensive models, local verification on affected tensors is the practical default.
Use full end-to-end validation as the final regression gate.

### Q2: Why keep both fused exports?
- Compute-graph export is convenient for tool-side analysis.
- Full-graph export is the safer choice for deployment/runtime loading.


## Summary

This notebook provides a reusable YOLOv5 Conv+Silu fusion workflow in ONNX-Tool, including:
1. Pattern search and fusion for `Conv + Sigmoid + Mul -> Conv_Silu`.
2. Local tensor-level verification template.
3. Optional end-to-end verification.
4. Export of both analysis and deployment-oriented fused models.
